# Text Preprocessing and Data Splitting

Notebook ini melakukan pembersihan teks, label encoding, dan pembagian dataset menjadi train, validation, dan test set.

In [1]:
import pandas as pd
import re
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## 1. Load Dataset

In [2]:
df = pd.read_csv('../data/bisniscom_2021-2022.csv', names=['category', 'date', 'title', 'image_url', 'url', 'content'])
df['text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')

## 2. Text Cleaning

Fungsi `clean_text` ini menghapus URL, karakter aneh, namun menjaga sebagian besar teks utuh (cocok untuk Deep Learning & IndoBERT).

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text) # Hapus URL
    text = re.sub(r'[^a-z0-9\s]', ' ', text) # Hapus karakter non-alfanumerik
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi berlebih
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()

,text,clean_text
0,"Ganti Tahun, Stok Pupuk Nasional di Atas Keten...",ganti tahun stok pupuk nasional di atas ketent...
1,"Pasar Bebas (AfCFTA) di Afrika Berlaku, Total ...",pasar bebas afcfta di afrika berlaku total nil...
2,"Wah, Pendengar Podcast Meningkat 3 Kali Lipat ...",wah pendengar podcast meningkat 3 kali lipat b...
3,Pelaku Pariwisata Nilai Pelarangan WNA Masuk R...,pelaku pariwisata nilai pelarangan wna masuk r...
4,Satgas Karantina 2.000 Penumpang WNA dan WNI B...,satgas karantina 2 000 penumpang wna dan wni b...


## 3. Label Encoding

In [4]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Mapping Label:", label_mapping)

# Simpan Label Encoder
os.makedirs('../models', exist_ok=True)
joblib.dump(le, '../models/label_encoder.pkl')

Mapping Label: {'APBN': np.int64(0), 'Agribisnis': np.int64(1), 'Ekonomi': np.int64(2), 'Ekonomi Global': np.int64(3), 'Energi & Tambang': np.int64(4), 'Infrastruktur': np.int64(5), 'Jasa & Niaga': np.int64(6), 'Manufaktur': np.int64(7), 'Pajak': np.int64(8), 'Properti': np.int64(9), 'Transportasi & Logistik': np.int64(10)}


['../models/label_encoder.pkl']

## 4. Train-Validation-Test Split (70:15:15)

In [5]:
# Pertama, split Train (70%) dan Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['clean_text'], df['label'], test_size=0.3, random_state=42, stratify=df['label']
)

# Kedua, split Temp menjadi Validation (15%) dan Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

Train size: 20687
Validation size: 4433
Test size: 4434


## 5. Simpan Data Split

In [6]:
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df = pd.DataFrame({'text': X_val, 'label': y_val})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_df.to_csv('../data/train.csv', index=False)
val_df.to_csv('../data/val.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

print("Data split berhasil disimpan!")

Data split berhasil disimpan!
